<a href="https://colab.research.google.com/github/NJerez-dev/Logistics-data-portfolio/blob/main/inventory_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generación Dataset

# Load Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:

np.random.seed(42)
# parámetros base
n_products = 50
n_days = 180
warehouses = ['Santiago', 'Valparaiso', 'Concepcion']


In [ ]:
# crear catálogo productos
products = pd.DataFrame({
    'product_id': range(1, n_products+1),
    'product_name': [f'Product_{i}' for i in range(1, n_products+1)],
    'category': np.random.choice(
        ['Electronics','Home','Industrial','Food'],
        n_products
    ),
    'unit_price': np.random.randint(5000, 50000, n_products),
    'reorder_point': np.random.randint(20, 100, n_products),
    'supplier_lead_time': np.random.randint(3, 15, n_products)
})

In [ ]:
# generar fechas
dates = pd.date_range(start='2024-01-01', periods=n_days)

data = []

for date in dates:
   for _, product in products.iterrows():

        units_sold = np.random.poisson(5)
        stock_level = max(0, np.random.normal(100, 30))

        warehouse = np.random.choice(warehouses)

        revenue = units_sold * product['unit_price']

        data.append([
            date,
            product['product_id'],
            product['product_name'],
            product['category'],
            warehouse,
            units_sold,
            product['unit_price'],
            revenue,
            int(stock_level),
            product['reorder_point'],
            product['supplier_lead_time']
        ])

columns = [
'date','product_id','product_name','category','warehouse',
'units_sold','unit_price','revenue','stock_level',
'reorder_point','supplier_lead_time']

df = pd.DataFrame(data, columns=columns)

df.head()

In [ ]:
df.to_csv('inventory_transactions.csv', index=False)

# Load Dataset


In [ ]:
df = pd.read_csv("inventory_transactions.csv")

df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

# Análisis de ventas

Ventas totales

In [ ]:
total_revenue = df['revenue'].sum()
total_units = df['units_sold'].sum()

print("Total Revenue:", total_revenue)
print("Total Units Sold:", total_units)

Ventas por categoría

In [ ]:
revenue_category = df.groupby("category")["revenue"].sum().sort_values(ascending=False)

revenue_category.plot(kind="bar")
plt.title("Revenue by Category")
plt.ylabel("Revenue")
plt.show()

Ventas por bodega

In [ ]:
warehouse_sales = df.groupby("warehouse")["revenue"].sum()

warehouse_sales.plot(kind="bar")
plt.title("Revenue by Warehouse")
plt.show()

# Top productos por margen


In [ ]:
top_products = df.groupby("product_name")["revenue"].sum().sort_values(ascending=False).head(10)

top_products.plot(kind="barh")
plt.title("Top 10 Products by Revenue")
plt.show()

# Análisis de inventario

Stock Promedio

In [ ]:
avg_stock = df.groupby("product_name")["stock_level"].mean()

avg_stock.head()

Productos con riesgo de quiebre

In [ ]:
df["stockout_risk"] = df["stock_level"] < df["reorder_point"]

stockout_products = df[df["stockout_risk"] == True]

stockout_products.head()

# Business Insights

1. La categoría Industrial genera el mayor volumen de ingresos.
2. La bodega de Santiago concentra la mayor demanda.
3. Se detectaron múltiples casos donde el stock cae bajo el punto de reposición.
4. Esto sugiere oportunidades para mejorar políticas de reabastecimiento.

## KPIs de Gestión de Inventarios por Producto y Bodega

In [ ]:


daily_demand_pw = df.groupby(['product_id', 'warehouse', 'date'])['units_sold'].sum().reset_index()

product_warehouse_demand_stats = daily_demand_pw.groupby(['product_id', 'warehouse'])['units_sold'].agg([
    'mean',
    'std'
]).reset_index()

product_warehouse_demand_stats.rename(columns={
    'mean': 'avg_daily_demand_pw',
    'std': 'std_dev_daily_demand_pw'
}, inplace=True)

# Unir con detalles del producto (supplier_lead_time, category)
# Asumimos que supplier_lead_time y category son los mismos para un product_id, independientemente de la bodega
product_details_for_merge = df[['product_id', 'category', 'supplier_lead_time']].drop_duplicates()
warehouse_product_kpis = pd.merge(product_warehouse_demand_stats, product_details_for_merge, on='product_id')

# Definir Z-score para un nivel de servicio deseado (ej., 95% de nivel de servicio = Z-score de 1.645)
Z = 1.645

# Calcular Stock de Seguridad (Safety Stock) por bodega
# SS = Z * std_dev_demand * sqrt(lead_time)
warehouse_product_kpis['safety_stock_pw'] = (Z * warehouse_product_kpis['std_dev_daily_demand_pw'] * np.sqrt(warehouse_product_kpis['supplier_lead_time'])).fillna(0).astype(int)

# Calcular Punto de Reorden (Reorder Point) por bodega
# ROP = (avg_daily_demand * lead_time) + Safety Stock
warehouse_product_kpis['reorder_point_pw'] = (warehouse_product_kpis['avg_daily_demand_pw'] * warehouse_product_kpis['supplier_lead_time'] + warehouse_product_kpis['safety_stock_pw']).astype(int)

print("KPIs de Gestión de Inventarios por Producto y Bodega (primeras 5 filas):")
display(warehouse_product_kpis.head())

## KPIs Agregados a Nivel de Producto


In [ ]:
# Agregar la demanda de cada producto a través de todas las bodegas para
# obtener un KPI a nivel de producto. Esto produce el DataFrame
# `product_kpis` que se utiliza en la visualización tipo semáforo.

daily_demand_p = (
    df.groupby(['product_id', 'date'])['units_sold']
    .sum()
    .reset_index()
)

product_demand_stats = (
    daily_demand_p.groupby('product_id')['units_sold']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={
        'mean': 'avg_daily_demand',
        'std': 'std_dev_daily_demand',
    })
)

product_kpis = pd.merge(
    product_demand_stats, product_details_for_merge, on='product_id'
)

product_kpis['safety_stock'] = (
    Z
    * product_kpis['std_dev_daily_demand']
    * np.sqrt(product_kpis['supplier_lead_time'])
).fillna(0).astype(int)

product_kpis['reorder_point_calculated'] = (
    product_kpis['avg_daily_demand'] * product_kpis['supplier_lead_time']
    + product_kpis['safety_stock']
).astype(int)

print('KPIs a nivel de producto (primeras 5 filas):')
display(product_kpis.head())


## Visualización de Cantidad Óptima (Punto de Reorden y Stock de Seguridad) para Productos Top, Medios y Bajos

In [ ]:


df = pd.read_csv("inventory_transactions.csv")

# 1. Get the latest stock level for each product (assuming 'date' column indicates temporal order)
# We'll use the stock level from the latest date for each product as its 'current' stock.
latest_stock_level_per_product = df.groupby('product_id').apply(lambda x: x.loc[x['date'].idxmax()][['stock_level']]).reset_index()
latest_stock_level_per_product.rename(columns={'stock_level': 'latest_current_stock'}, inplace=True)

# 2. Merge with product_kpis
product_kpis_with_stock = pd.merge(product_kpis, latest_stock_level_per_product, on='product_id')

# Merge product names for visualization
product_names = df[['product_id', 'product_name']].drop_duplicates()
product_kpis_with_stock = pd.merge(product_kpis_with_stock, product_names, on='product_id')

# 3. Define inventory risk categories (Traffic Light)
# The 'reorder_point_calculated' already includes safety stock (ROP = demand_during_lead_time + SS).
# We define risk based on the latest current stock relative to this ROP and an additional buffer.

buffer_days = 7 # Define a buffer in terms of days of average daily demand for the 'Yellow' to 'Green' threshold

def assign_inventory_risk_traffic_light(row):
    reorder_point = row['reorder_point_calculated']
    avg_daily_demand = row['avg_daily_demand']
    current_stock = row['latest_current_stock']

    # Define the threshold for 'Green' status: ROP + (buffer_days of avg_daily_demand)
    # This buffer ensures that even if above ROP, if it's not significantly above, it's 'Yellow'.
    buffer_threshold = reorder_point + (avg_daily_demand * buffer_days)

    if current_stock < reorder_point:
        return 'Rojo (Crítico: Bajo ROP)'
    elif current_stock < buffer_threshold:
        return 'Amarillo (Alerta: Cerca de ROP)'
    else:
        return 'Verde (Saludable: Buen Stock)'

product_kpis_with_stock['inventory_risk_traffic_light'] = product_kpis_with_stock.apply(assign_inventory_risk_traffic_light, axis=1)

# Define color palette for traffic light
traffic_light_palette = {
    'Rojo (Crítico: Bajo ROP)': 'red',
    'Amarillo (Alerta: Cerca de ROP)': 'gold',
    'Verde (Saludable: Buen Stock)': 'limegreen'
}

# Sort products by risk level (Red, Yellow, Green) and then by latest_current_stock for better visualization
risk_order_numeric_map = {
    'Rojo (Crítico: Bajo ROP)': 1,
    'Amarillo (Alerta: Cerca de ROP)': 2,
    'Verde (Saludable: Buen Stock)': 3
}
product_kpis_with_stock['risk_order_numeric'] = product_kpis_with_stock['inventory_risk_traffic_light'].map(risk_order_numeric_map)
product_kpis_with_stock_sorted = product_kpis_with_stock.sort_values(
    by=['risk_order_numeric', 'latest_current_stock'], ascending=[True, True]
)

# 4. Create the Traffic Light style visualization
fig, ax = plt.subplots(figsize=(18, 10))

# Create bars for latest current stock, colored by risk
bars = ax.bar(
    product_kpis_with_stock_sorted['product_name'],
    product_kpis_with_stock_sorted['latest_current_stock'],
    color=[traffic_light_palette[risk] for risk in product_kpis_with_stock_sorted['inventory_risk_traffic_light']],
    label='Stock Actual (Último Registro)'
)

# Add horizontal lines for Reorder Point for each product
ax.hlines(
    y=product_kpis_with_stock_sorted['reorder_point_calculated'],
    xmin=np.arange(len(product_kpis_with_stock_sorted)), # X-coordinates for each bar
    xmax=np.arange(len(product_kpis_with_stock_sorted)) + 0.8, # Extend line slightly right from bar center
    colors='blue',
    linestyle='-', # Solid line for Reorder Point
    linewidth=2,
    label='Punto de Reorden Calculado'
)

# Add horizontal lines for the Yellow/Green buffer threshold
buffer_threshold_values = product_kpis_with_stock_sorted['reorder_point_calculated'] + (product_kpis_with_stock_sorted['avg_daily_demand'] * buffer_days)
ax.hlines(
    y=buffer_threshold_values,
    xmin=np.arange(len(product_kpis_with_stock_sorted)),
    xmax=np.arange(len(product_kpis_with_stock_sorted)) + 0.8,
    colors='darkgreen',
    linestyle='--', # Dashed line for Healthy Stock Threshold
    linewidth=2,
    label=f'Umbral de Stock Saludable (ROP + {buffer_days} días de demanda promedio)'
)

ax.set_title('Estado de Inventario por Producto', fontsize=16)
ax.set_xlabel('Producto', fontsize=12)
ax.set_ylabel('Cantidad de Unidades', fontsize=12)
plt.xticks(rotation=90, ha='right', fontsize=10) # Rotate x-axis labels for readability
ax.ticklabel_format(style='plain', axis='y') # Avoid scientific notation on y-axis
ax.legend(loc='upper left', bbox_to_anchor=(1, 1)) # Place legend outside the plot to avoid overlap
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()

# Display a table with key values for context
print("\nDetalle del Estado de Inventario por Producto (primeras 10 filas):")
display(product_kpis_with_stock_sorted[[
    'product_name', 'latest_current_stock', 'reorder_point_calculated', 'safety_stock',
    'avg_daily_demand', 'supplier_lead_time', 'inventory_risk_traffic_light'
]].head(10))


In [ ]:
category_warehouse_inventory = warehouse_product_kpis.groupby(['category', 'warehouse']).agg(
    total_reorder_point=('reorder_point_pw', 'sum'),
    total_safety_stock=('safety_stock_pw', 'sum')
).reset_index()

# Create a combined label for the x-axis
category_warehouse_inventory['category_warehouse_label'] = category_warehouse_inventory['category'] + ' - ' + category_warehouse_inventory['warehouse']

fig, ax = plt.subplots(figsize=(15, 8))
category_warehouse_inventory.set_index('category_warehouse_label')[['total_reorder_point', 'total_safety_stock']].plot(kind='bar', stacked=True, ax=ax)

ax.set_title('Punto de Reorden y Stock de Seguridad por Categoría y Bodega', fontsize=14)
ax.set_xlabel('Categoría - Bodega', fontsize=12)
ax.set_ylabel('Cantidad de Unidades', fontsize=12)
ax.ticklabel_format(style='plain', axis='y')
plt.xticks(rotation=90, ha='right', fontsize=10) # Rotate labels for readability
plt.legend(title='Tipo de Stock')
plt.tight_layout()
plt.show()

print("Punto de Reorden y Stock de Seguridad por Categoría y Bodega:")
display(category_warehouse_inventory[['category', 'warehouse', 'total_reorder_point', 'total_safety_stock']])

### Detalle de Punto de Reorden y Stock de Seguridad para un Producto Específico por Bodega



In [ ]:
product_to_analyze = 12 # Puedes cambiar este ID para analizar otro producto
product_sample_kpis = warehouse_product_kpis[warehouse_product_kpis['product_id'] == product_to_analyze]

if not product_sample_kpis.empty:
    fig, ax = plt.subplots(figsize=(12, 7))
    product_sample_kpis.set_index('warehouse')[['reorder_point_pw', 'safety_stock_pw']].plot(kind='bar', stacked=True, ax=ax, color=['lightgreen', 'orange'])
    ax.set_title(f'Punto de Reorden y Stock de Seguridad para Product_{product_to_analyze} por Bodega')
    ax.set_xlabel('Bodega')
    ax.set_ylabel('Cantidad de Unidades')
    ax.ticklabel_format(style='plain', axis='y')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    print(f"Detalle de KPIs para Product_{product_to_analyze}:")
    display(product_sample_kpis[['warehouse', 'avg_daily_demand_pw', 'supplier_lead_time', 'safety_stock_pw', 'reorder_point_pw']])
else:
    print(f"Product_{product_to_analyze} no encontrado en warehouse_product_kpis.")

## Exportación de Resultados a Excel

Todas las tablas y los principales gráficos se consolidan en un libro
Excel multi-hoja (`inventario_kpis.xlsx`) para facilitar su revisión por
stakeholders no técnicos.


In [ ]:
import os

output_path = 'inventario_kpis.xlsx'

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name='Transacciones', index=False)
    revenue_category.to_frame('revenue').to_excel(
        writer, sheet_name='Ingresos_por_Categoria'
    )
    warehouse_sales.to_frame('revenue').to_excel(
        writer, sheet_name='Ingresos_por_Bodega'
    )
    top_products.to_frame('revenue').to_excel(
        writer, sheet_name='Top_Productos'
    )
    product_kpis.to_excel(
        writer, sheet_name='KPIs_Producto', index=False
    )
    warehouse_product_kpis.to_excel(
        writer, sheet_name='KPIs_Producto_Bodega', index=False
    )
    category_warehouse_inventory.to_excel(
        writer, sheet_name='KPIs_Categoria_Bodega', index=False
    )

print(f'Reporte Excel generado: {os.path.abspath(output_path)}')
